In [16]:
import pandas as pd

løn = pd.read_csv("data/løn.csv", encoding="latin1", sep=";", header=None)

print(løn.head())

     0      1                                        2              3   \
0  Alle  I alt  Gennemsnit for personer i gruppen (kr.)      København   
1  Alle  I alt  Gennemsnit for personer i gruppen (kr.)  Frederiksberg   
2  Alle  I alt  Gennemsnit for personer i gruppen (kr.)         Dragør   
3  Alle  I alt  Gennemsnit for personer i gruppen (kr.)         Tårnby   
4  Alle  I alt  Gennemsnit for personer i gruppen (kr.)    Albertslund   

       4       5       6       7       8       9   ...      27      28  \
0  144390  146145  157207  161926  167559  172856  ...  304699  311523   
1  174495  179198  189530  197266  204176  212146  ...  364056  371637   
2  203082  206589  211044  220330  231570  239897  ...  395557  399826   
3  170590  171550  179304  184378  190478  195427  ...  310723  315828   
4  163585  165338  173912  179120  183150  187986  ...  275786  280298   

       29      30      31      32      33      34      35      36  
0  319745  329200  342385  358403  380513 

In [17]:
def clean_income(df):
    df = df.copy()

    # Rename kommune column
    df = df.rename(columns={df.columns[3]: "kommune"})

    # Identify year columns
    value_cols = list(df.columns[4:])
    n_years = len(value_cols)

    start_year = 1992
    years = list(range(start_year, start_year + n_years))

    # Rename year columns
    df = df.rename(columns=dict(zip(value_cols, years)))

    # Clean kommune names
    df["kommune"] = (
        df["kommune"]
        .astype(str)
        .str.strip()
        .str.replace("Lyngby-Tårbæk", "Lyngby-Taarbæk", regex=False)
        .str.replace("Vesthimmerlands", "Vesthimmerland", regex=False)
    )

    # Melt to long format
    df = df.melt(
        id_vars=["kommune"],
        value_vars=years,
        var_name="år",
        value_name="income"
    )

    # Convert types
    df["år"] = df["år"].astype(int)
    df["income"] = pd.to_numeric(df["income"], errors="coerce")

    # Filter relevant years
    df = df[(df["år"] >= 1992) & (df["år"] <= 2024)].copy()

    # Drop non-municipal rows
    drop_rows = [
        "Hele landet",
        "København og Frederiksberg",
        "Christiansø"
    ]

    df = df[~df["kommune"].isin(drop_rows)]

    return df

In [19]:
income_clean = clean_income(løn)

print(income_clean.head())
print(income_clean["år"].min(), income_clean["år"].max())
print(income_clean["kommune"].nunique())

income_clean.to_csv("data/cleaned/income.csv", index=False)

         kommune    år  income
0      København  1992  144390
1  Frederiksberg  1992  174495
2         Dragør  1992  203082
3         Tårnby  1992  170590
4    Albertslund  1992  163585
1992 2024
98


In [20]:
migration = pd.read_csv("data/cleaned/migration_population_clean.csv")
housing = pd.read_csv("data/cleaned/ejendomsdata.csv")

#df = migration.merge(income_clean, on=["kommune", "år"], how="left")
#df = df.merge(housing, on=["kommune", "år"], how="left")